In [71]:
import numpy as np
import pandas as pd
import cvxpy as cp
import seaborn as sns
import mosek
import matplotlib.pyplot as plt
import datetime as date
from datetime import datetime as dt
from dateutil.relativedelta import *
import scipy.stats
from scipy.stats import rankdata
import itertools
#from itertools import chain, combinations
from Hit_and_Run import hit_and_run
import affine_approx as af

In [59]:
def argmax_sing_power(x2,x1,par):
    breuk = ((1-x1)**par-(1-x2)**par)/(x2-x1)
    return(1-(breuk/par)**(1/(par-1)))

def max_error_singpw(x2,x1,x,par):
    breuk = ((1-x1)**par-(1-x2)**par)/(x2-x1)
    return((1-x1)**par-(1-x)**par-breuk*(x-x1))

def sing_pw(x,par):
    return(1-(1-x)**par)

In [13]:
def kullback_leibler(p,q,r,par,constraints):
    N = p.shape[0]
    phi_cons = 0
    for i in range(N):
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons<=r)
    return(constraints)


def kullback_leibler_conj(gamma,w,s,t,constraints):
    N = s.shape[0]
    constraints.append(w - gamma*(np.zeros(N)+1) <= t)
    for i in range(N):
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
    return(constraints)

def h_single_power(x,par):
    return(1-(np.abs(1-x))**par)

def h_sing_power(q_b,q,rank,par,constraints):
    N = q.shape[0]
    for i in range(N-1):
        v = 1-cp.power((1-cp.sum(q[rank[0:i+1]])),par)
        constraints.append(cp.sum(q_b[rank[0:i+1]])-v <= 0)
    return(constraints)

def h_sing_power_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    xi_2 = cp.Variable(M, nonneg = True)
    xi_3 = cp.Variable(M, nonneg = True)
    xi_4 = cp.Variable(M, nonneg = True)
    constraints.extend((xi_3 <= xi_2, xi_3 <= v))
    constraints.append(lbda-xi_3+(par**(-1/(par-1))-par**(-par/(par-1)))*xi_4 <= z)
    exponent = np.array([(par-1)/par,1-(par-1)/par])
    for j in range(M):
        constraints.append(xi_2[j]-cp.geo_mean(cp.vstack([xi_4[j],lbda[j]]),exponent)<= 0)
    return(constraints)

def h_pow(x,par):
    return(x**par)

def h_power(q_b,q,rank,par,constraints):
    N = q.shape[0]
    for i in range(N-1):
        constraints.append(cp.sum(q_b[rank[0:i+1]])-cp.power(cp.sum(q[rank[0:i+1]]),par) <= 0)
    return(constraints)

def h_power_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    constraints.append(v >= 0)
    const = (par**(par/(1-par))-par**(1/(1-par)))**(1-par)
    exp = np.array([1-par, par])
    for j in range(M):
        constraints.append(cp.geo_mean(cp.vstack([z[j],v[j]]),exp)>= lbda[j]*const)
    return(constraints)

def h_cv(x, par):
    return(np.minimum(x/(1-par),1))

def h_cvar(q_b, q, rank, par, constraints):
    constraints.append(q_b<= q/(1-par))
    return(constraints)

def h_cvar_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    for j in range(M):
        constraints.append(cp.pos(-(1-par)*v[j]+lbda[j])<= z)
    constraints.append(v >= 0)
    return(constraints)

In [68]:
d = np.array([4,8,10])
p_items = np.array([[0.375,0.375,0.25],[0.25,0.25,0.5],[0.127,0.786,0.087]])
par_mnews = np.array([[6,4,2,4],[8,3,5,2.5],[5,4,1.5,4]])
d_item = len(p_items)
d_demand = len(d)
m = d_demand**d_item
N = 1000
par = 2
phi_dot = 1
r = phi_dot/(2*N)*scipy.stats.chi2.ppf(0.95, N)
phi_conj = kullback_leibler_conj
h_conj = h_sing_power_conj
phi_func = kullback_leibler
h_func = h_sing_power
h_eva = h_single_power
eps = 0.001

In [57]:
indices = np.asarray(list((itertools.product((0, 1, 2), repeat = 3))))
p = np.zeros(m)
for i in range(m):
    p[i] = np.prod(p_items[np.arange(d_item),indices[i]])

1.0

In [61]:
def r_obj(d,y,par_news):
    [v_news, l_news, s_news, c_news] = par_news
    return((s_news-v_news)*cp.pos(y-d)-l_news*cp.pos(d-y)+(v_news-c_news)*y)

In [65]:
def mnews_affine_riskmin(p,d,r,indices,par,par_mnews,phi_conj,slope,const):
    N = len(p)
    I = len(par_mnews)
    K = len(slope)
    lbda = cp.Variable((N,K), nonneg = True)
    y = cp.Variable(I, nonneg = True)
    v = cp.Variable(K, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    s = cp.Variable(N)
    c = cp.Variable(1)
    constraints = [y<= np.max(d)]
    profit = np.zeros(N)
    for i in range(N):
        for j in range(I):
            profit[i] = profit[i] + r_obj(d[indices[i][j]],y[j],par_mnews[j])
    for i in range(N):
        constraints.append(-profit[i] - cp.sum(lbda[i]) - beta <= 0)
        constraints.append(s[i] == -alpha + lbda[i]@slope)
        constraints.append(lbda[i] <= v)
    constraints = phi_conj(gamma,s,t,constraints)
    constraints.append(alpha + beta + gamma * r  + v@const + p@t <= c)
    obj = cp.Maximize(-c)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(y.value, -prob.value)


def mnews_rc_riskmin(sets,p,d,r,par,par_mnews,indices,phi_conj, h_conj):
    N, I, M = [len(p), len(par_mnews), len(sets)]
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    y = cp.Variable(I, nonneg = True)
    alpha, beta = [cp.Variable(1), cp.Variable(1)]
    gamma = cp.Variable(1,nonneg = True)
    t, z, s = [cp.Variable(N),cp.Variable(N),cp.Variable(N)]
    constraints = [y<= np.max(d)]
    profit = np.zeros(N)
    for i in range(N):
        for j in range(I):
            profit[i] = profit[i] + r_obj(d[indices[i][j]],y[j],par_mnews[j])
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append(-profit[i] - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
    constraints = phi_conj(gamma,s,t,constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize(-c)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(y.value, -prob.value)

In [72]:
x_points = af.affine_approx(eps,argmax_sing_power,max_error_singpw,par)

In [73]:
x_points

array([0.        , 0.06347656, 0.12695312, 0.19014502, 0.25313568,
       0.31668486, 0.38004427, 0.44347185, 0.5065782 , 0.56956026,
       0.63282479, 0.696086  , 0.75904127, 0.82251897, 0.88563873,
       0.94869689, 1.        ])